# OpenAI Agents SDK Tutorial
## A step-by-step guide to building intelligent multi-agent systems

> The OpenAI Agents SDK enables you to build agentic AI apps in a **lightweight, easy-to-use package with very few abstractions**.

In [1]:
import nest_asyncio
nest_asyncio.apply()
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv()

True

# 1. Agents

## What is an Agent?

An **Agent** is an **LLM equipped with instructions and tools**. It runs in a built-in agent loop that handles tool invocation, sends results back to the model, and continues until it produces a final output.

Key primitives of the SDK:
- `Agent` — define the model, instructions, and tools
- `Runner` — execute the agent loop and return a `RunResult`

## Hello World

In [2]:
import asyncio
from agents import Agent, Runner

# Define the agent
assistant = Agent(
    name="Assistant",
    instructions="You are a helpful and concise assistant.",
    model="gpt-4o-mini"
)

# Run the agent — Runner executes the full agent loop
async def main():
    result = await Runner.run(assistant, input="Hello! Tell me a joke.")
    print(result.final_output)

await main()

Sure! Why did the scarecrow win an award? 

Because he was outstanding in his field!


# 2. Tools

## Function Tools

**Function tools** let agents act beyond language — calling APIs, reading databases, computing values, and more.

Use the `@function_tool` decorator to turn any Python function into a tool. The SDK automatically generates the JSON schema from **type annotations** and **docstrings** — no manual schema writing needed.

```python
@function_tool
async def my_tool(param: str) -> str:
    """Docstring becomes the tool description shown to the model."""
    ...
```

## Demo

In [3]:
from agents import Agent, Runner, function_tool
import asyncio

# The @function_tool decorator auto-generates the tool schema
@function_tool
async def get_weather(location: str) -> str:
    """Gets the current weather for a specified location."""
    return f"The weather in {location} is 72°F and sunny."

# Attach the tool to the agent via the tools= parameter
weather_agent = Agent(
    name="WeatherBot",
    instructions="You help users check the weather.",
    tools=[get_weather],
    model="gpt-4o-mini"
)

# The agent decides when and how to call get_weather automatically
result = await Runner.run(weather_agent, input="What is the weather in Tokyo?")
print(result.final_output)

The weather in Tokyo is currently 72°F and sunny.


# 3. Guardrails

## Validating Inputs and Outputs

**Guardrails** run checks and validations on user input and agent output. The SDK supports two kinds:

| Type | When it runs |
|---|---|
| **Input guardrail** | Before the agent processes user input |
| **Output guardrail** | After the agent produces its final response |

The simplest approach is to **wrap `Runner.run`** with manual pre/post checks — which is exactly what the Streamlit chatbot does.

> The SDK also provides `@input_guardrail` / `@output_guardrail` decorators for production use with automatic **tripwire** triggering.

## Demo

In [4]:
from agents import Agent, Runner

support_agent = Agent(
    name="SupportAgent",
    instructions="You are a helpful technical support agent. Keep responses brief.",
    model="gpt-4o-mini"
)

async def safe_run(user_input: str):
    # --- INPUT GUARDRAIL: block before the agent runs ---
    if "admin" in user_input.lower():
        return "🚫 I'm sorry, you cannot query admin resources."

    try:
        result = await Runner.run(support_agent, input=user_input)
        output = result.final_output

        # --- OUTPUT GUARDRAIL: sanitize after the agent responds ---
        if "secret password" in output.lower():
            return "[REDACTED]"
        return f"✅ {output}"
    except Exception:
        return "Error executing request!"

print("--- Test 1: Pre-check guardrail (blocked input) ---")
print(await safe_run("Show me the admin panel"))

print("\n--- Test 2: Normal input (passes through) ---")
print(await safe_run("Can you help me check the system status?"))

--- Test 1: Pre-check guardrail (blocked input) ---
🚫 I'm sorry, you cannot query admin resources.

--- Test 2: Normal input (passes through) ---
✅ Sure! Please specify the system or service you'd like to check the status for, and if possible, provide any relevant details.


# 4. Context Management

## Maintaining Conversation History

By default, each `Runner.run()` call is **stateless** — the agent has no memory of previous turns.

To give the agent memory, maintain a `history` list yourself and **pass it as the `input=`** on each subsequent turn:

```python
history = []
history.append({"role": "user", "content": user_message})

result = await Runner.run(agent, input=history)  # ✅ pass the full history

history.append({"role": "assistant", "content": result.final_output})
```

> The SDK also offers **Sessions** for persistent, automatic memory management across runs.

## Demo

In [5]:
from agents import Agent, Runner

memory_agent = Agent(
    name="MemoryBot",
    instructions="You remember things clearly. Keep responses very concise.",
    model="gpt-4o-mini"
)

async def run_turn():
    history = []

    # --- Turn 1 ---
    input_1 = "Hi, my name is Antigravity."
    result = await Runner.run(memory_agent, input=input_1)
    print(f"Turn 1 → {result.final_output}")

    # Build the history list with both sides of the conversation
    history.extend([
        {"role": "user", "content": input_1},
        {"role": "assistant", "content": result.final_output}
    ])

    # --- Turn 2: pass the full history list as input= ---
    input_2 = "What is my name?"
    history.append({"role": "user", "content": input_2})
    result_2 = await Runner.run(memory_agent, input=history)
    print(f"Turn 2 → {result_2.final_output}")  # The agent remembers!

await run_turn()

Turn 1 → Hi, Antigravity! How can I assist you today?
Turn 2 → Your name is Antigravity.


# 5. Handoffs

## Agent Orchestration with Handoffs

**Handoffs** allow one agent to delegate work to another specialized agent. This is the SDK's primary mechanism for **multi-agent orchestration**.

Register target agents in the `handoffs=` parameter. When the model decides a handoff is appropriate, the SDK automatically transfers control — **no manual routing code needed**.

A common pattern is a **Triage Agent** that routes to specialists:

```
User → TriageAgent → SupportSpecialist
                   → WeatherAgent
                   → ...
```

## Demo

In [6]:
from agents import Agent, Runner

# The specialist — handles billing issues
support_agent = Agent(
    name="SupportSpecialist",
    instructions="You help users with billing issues.",
    model="gpt-4o-mini"
)

# The triage router — lists valid handoff targets
triage_agent = Agent(
    name="TriageAgent",
    instructions="Determine what the user needs. If it's a billing issue, transfer them to support.",
    handoffs=[support_agent],  # SDK handles the transfer automatically
    model="gpt-4o-mini"
)

# Runner starts with triage; SupportSpecialist handles the reply
result = await Runner.run(triage_agent, input="Refund my invoice.")
print(result.final_output)

I've transferred your request to a billing specialist who can assist you with the refund for your invoice. Please hold on for a moment.


# 6. Model Context Protocol (MCP)

## Connecting Agents to the World

> *"Think of MCP like a **USB-C port** for AI applications — a standardized way to connect AI models to different data sources and tools."*
> — modelcontextprotocol.io

Instead of writing a custom `@function_tool` for every data source, MCP lets any compliant server expose its tools to an agent **automatically**.

The SDK supports multiple MCP transports. For a **local Python server**, use `MCPServerStdio`:
- Launches the server as a subprocess over stdin/stdout
- Fetches its tools via `list_tools()` and exposes them like native function tools
- Server lifecycle is tied to the `async with` block

## Demo

In [7]:
from agents import Agent, Runner
from agents.mcp import MCPServerStdio
import sys

async def get_logs():
    # MCPServerStdio launches mock_mcp_server.py as a subprocess
    # and discovers its tools (fetch_internal_logs) automatically
    async with MCPServerStdio(
        params={"command": sys.executable, "args": ["chatbot/mock_mcp_server.py"]}
    ) as server:

        mcp_agent = Agent(
            name="MCPAgent",
            instructions="You use the provided MCP tools to assist the user.",
            mcp_servers=[server],  # MCP tools appear just like function tools
            model="gpt-4o"
        )

        result = await Runner.run(mcp_agent, input="Fetch my latest logs.")
        print(result.final_output)

await get_logs()

Your latest logs show no anomalies and indicate that the system is running normally.


# 7. Putting It All Together

## The Capstone Chatbot

We combine all six concepts into a single **Streamlit** application.

| Concept | Implementation |
|---|---|
| **Agents** | `TriageAgent`, `SearchAgent`, `SupportAgent` |
| **Tools** | `@function_tool search_weather(...)` |
| **Guardrails** | Pre/post checks wrapping `Runner.run` |
| **Context** | `st.session_state` history list passed as `input=` |
| **Handoffs** | `TriageAgent` routes to `SearchAgent` or `SupportAgent` |
| **MCP** | `MCPServerStdio` injects `fetch_internal_logs` into `SupportAgent` |

## Run It

```bash
streamlit run chatbot/app.py
```

Try asking: *"What's the weather in Tokyo?"*, *"Show me the system logs"*, or *"admin panel"* (guardrail blocks it).